In [ ]:
from langchain.memory import ChatMessageHistory

# 初始化历史对象
history = ChatMessageHistory()

# 添加用户和AI消息
history.add_user_message("I'm heading to New York next week.")
history.add_ai_message("Great! It's a fantastic city.")

# 访问消息列表
print(history.messages)

In [ ]:
from langchain.memory import ConversationBufferMemory

# 初始化内存
memory = ConversationBufferMemory()

# 保存对话轮次
memory.save_context({"input": "What's the weather like?"}, {"output": "It's sunny today."})

# 将内存作为字符串加载
print(memory.load_memory_variables({}))

{'history': "Human: What's the weather like?\nAI: It's sunny today."}


/tmp/ipython-input-1-1418393889.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory()


In [ ]:
from langchain_openai import OpenAI
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain.memory import ConversationBufferMemory

# 1. 定义LLM和提示
llm = OpenAI(temperature=0)
template = """You are a helpful travel agent.

Previous conversation:
{history}

New question: {question}
Response:"""
prompt = PromptTemplate.from_template(template)

# 2. 配置内存
# memory_key“history”与提示中的变量匹配
memory = ConversationBufferMemory(memory_key="history")

# 3. 构建链条
conversation = LLMChain(llm=llm, prompt=prompt, memory=memory)

# 4. 进行对话
response = conversation.predict(question="I want to book a flight.")
print(response)
response = conversation.predict(question="My name is Sam, by the way.")
print(response)
response = conversation.predict(question="What was my name again?")
print(response)

ModuleNotFoundError: No module named 'langchain_openai'

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.chains import LLMChain
from langchain.memory import ConversationBufferMemory
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)

# 1. 定义聊天模型和提示
llm = ChatOpenAI()
prompt = ChatPromptTemplate(
    messages=[
        SystemMessagePromptTemplate.from_template("You are a friendly assistant."),
        MessagesPlaceholder(variable_name="chat_history"),
        HumanMessagePromptTemplate.from_template("{question}")
    ]
)

# 2. 配置内存
# return_messages=True 对于聊天模型至关重要
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# 3. 构建链条
conversation = LLMChain(llm=llm, prompt=prompt, memory=memory)

# 4. 进行对话
response = conversation.predict(question="Hi, I'm Jane.")
print(response)
response = conversation.predict(question="Do you remember my name?")
print(response)

ModuleNotFoundError: No module named 'langchain_openai'

In [4]:
# 更新代理指令的节点
def update_instructions(state: State, store: BaseStore):
    namespace = ("instructions",)
    # 从商店获取最新说明
    current_instructions = store.search(namespace)[0]

    # 创建一个提示，要求法学硕士反思对话
    # 并生成新的、改进的指令
    prompt = prompt_template.format(
        instructions=current_instructions.value["instructions"],
        conversation=state["messages"]
    )

    # 从 LLM 获取新说明
    output = llm.invoke(prompt)
    new_instructions = output['new_instructions']

    # 将更新后的说明保存回商店
    store.put(("agent_instructions",), "agent_a", {"instructions": new_instructions})


# 使用指令生成响应的节点
def call_model(state: State, store: BaseStore):
    namespace = ("agent_instructions", )
    # 从商店检索最新说明
    instructions = store.get(namespace, key="agent_a")[0]

    # 使用检索到的指令来格式化提示
    prompt = prompt_template.format(instructions=instructions.value["instructions"])
    # ...应用程序逻辑继续

NameError: name 'State' is not defined

In [ ]:
from langgraph.store.memory import InMemoryStore

# 真实嵌入函数的占位符
def embed(texts: list[str]) -> list[list[float]]:
    # 在实际应用中，使用适当的嵌入模型
    return [[1.0, 2.0] for _ in texts]

# 初始化内存存储。对于生产，请使用数据库支持的存储。
store = InMemoryStore(index={"embed": embed, "dims": 2})

# 为特定用户和应用程序上下文定义命名空间
user_id = "my-user"
application_context = "chitchat"
namespace = (user_id, application_context)

# 1.将内存放入store
store.put(
    namespace,
    "a-memory",  # 这段记忆的钥匙
    {
        "rules": [
            "User likes short, direct language",
            "User only speaks English & python",
        ],
        "my-key": "my-value",
    },
)

# 2. 通过namespace和key获取内存
item = store.get(namespace, "a-memory")
print("Retrieved Item:", item)

# 3. 在命名空间内搜索记忆，按内容过滤
# 并根据向量与查询的相似度进行排序。
items = store.search(
    namespace,
    filter={"my-key": "my-value"},
    query="language preferences"
)
print("Search Results:", items)